# Home Credit Default Risk
## Sprint 2 — Pré-Processamento e Feature Engineering

**Dataset:** `application_train.csv` (~307k linhas, 122 colunas)  
**Target:** `TARGET` — 0 = pagou normalmente, 1 = dificuldade de pagamento nas parcelas iniciais  
**Tipo de tarefa:** Classificação binária  

---

## SEÇÃO 1 — Revisão dos achados da Sprint 1

### 1.1 Breve recapitulação dos problemas identificados na EDA
Com base na análise realizada na Sprint 1, identificamos os seguintes pontos críticos no dataset `application_train.csv`:
* **Desbalanceamento:** O target é altamente desbalanceado (~92% pagaram normalmente e ~8% tiveram dificuldades).
* **Valores Ausentes:** 67 das 122 colunas possuem valores nulos. O problema é severo em dados de infraestrutura imobiliária (ex: `COMMONAREA_AVG` com 69,87%) e idade do veículo (`OWN_CAR_AGE` com 65,99%).
* **Fontes Externas:** As features de score externo têm grande poder preditivo, mas sofrem com ausências, especialmente a `EXT_SOURCE_1` (56,38% de nulos).
* **Variáveis para Transformação:** Temos 16 variáveis categóricas que precisarão de Encoding e 106 variáveis numéricas para avaliar escalonamento e outliers.

### 1.2 Lista de ações de pré-processamento planejadas
Para esta sprint, planejamos:
1. Dividir os dados em Treino e Teste antes de qualquer transformação para evitar *data leakage*.
2. Descartar colunas de infraestrutura com mais de 60% de nulos, pois não agregam valor e introduzem muito ruído.
3. Imputar valores nas variáveis críticas (como `EXT_SOURCE`) usando métodos estatísticos ou avançados.
4. Tratar outliers apenas nas variáveis financeiras em que a distorção afeta a modelagem.
5. Aplicar *One-Hot Encoding* em categóricas nominais e *Target Encoding* naquelas com muitas categorias.
6. Encapsular tudo em um Pipeline do Scikit-Learn.

In [ ]:
# 1.3 Carregamento do dataset e separação em Treino e Teste
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df_raw = pd.read_csv('../data/raw/application_train.csv')

# Separar treino e teste ANTES de qualquer transformação
X = df_raw.drop(columns=['TARGET', 'SK_ID_CURR']) 
y = df_raw['TARGET']

# Usando stratify=y devido ao forte desbalanceamento (8% da classe 1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Shape X_train: {X_train.shape}")
print(f"Shape X_test: {X_test.shape}")

---

## SEÇÃO 2 — Tratamento de dados ausêntes

### 2.1 Diagnóstico de missing values
Primeiro, vamos verificar a proporção exata de dados faltantes nas colunas da nossa base de treino.

In [ ]:
# Diagnóstico de nulos por coluna no treino
missing_cols = X_train.isnull().mean() * 100
missing_cols = missing_cols[missing_cols > 0].sort_values(ascending=False)
print("Top 10 colunas com mais valores ausentes (%):")
print(missing_cols.head(10))

# Diagnóstico de nulos por linha no treino
missing_rows = X_train.isnull().sum(axis=1)
print(f"\nMédia de valores ausentes por linha: {missing_rows.mean():.2f}")

### 2.2 Estratégias e Justificativas para Tratamento de Ausentes

Com base no diagnóstico, adotaremos as seguintes abordagens estruturadas:

* **Estratégia 1 - Remoção (Drop):** Colunas com mais de 60% de dados ausentes (ex: `COMMONAREA_AVG`, `OWN_CAR_AGE`) serão removidas. **Justificativa:** Imputar dados em variáveis com mais de 60% de ausência destrói a variância original e insere muito viés, pois significa inventar a maior parte da informação.
* **Estratégia 2 - Imputação Simples (Mediana/Moda):** Para variáveis com menos de 60% de nulos. **Justificativa:** A mediana será usada para números pois é robusta contra outliers. A Moda será usada para variáveis categóricas (textos).
* **Estratégia 3 - Imputação Avançada (IterativeImputer):** Para a `EXT_SOURCE_1` (56,38% de ausentes). **Justificativa:** Apesar da alta ausência, é a variável com maior poder preditivo do negócio. O `IterativeImputer` estimará esse valor cruzando dados de renda e outras fontes externas. *(Nota: Por padrão, aplicaremos a Mediana agora para garantir a execução limpa, mas a estrutura para o IterativeImputer ficará pronta para as próximas fases).*

### 2.3 Aplicação das Estratégias de Imputação

Abaixo, aplicamos as transformações definidas nas bases de Treino e Teste. Note que, para evitar o vazamento de dados (*data leakage*), os imputadores "aprendem" (`fit`) as métricas apenas na base de Treino e apenas aplicam (`transform`) na base de Teste.

In [ ]:
# 2.3 Aplicação das Estratégias de Imputação
from sklearn.impute import SimpleImputer
# from sklearn.experimental import enable_iterative_imputer
# from sklearn.impute import IterativeImputer

# Guardar cópia para evidência do antes e depois
X_train_before = X_train.copy()

# ESTRATÉGIA 1: Remoção de colunas com mais de 60% de nulos
cols_to_drop = missing_cols[missing_cols > 60.0].index.tolist()
X_train = X_train.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop) 
print(f"Foram removidas {len(cols_to_drop)} colunas por excesso de nulos.")

# ESTRATÉGIA 2: Imputação com a Mediana (Numéricas)
num_cols_to_impute = X_train.select_dtypes(include=[np.number]).columns
median_imputer = SimpleImputer(strategy='median')

# FIT apenas no treino para evitar data leakage
X_train[num_cols_to_impute] = median_imputer.fit_transform(X_train[num_cols_to_impute])
X_test[num_cols_to_impute] = median_imputer.transform(X_test[num_cols_to_impute])

# ESTRATÉGIA 3: Imputação com a Moda (Categóricas)
cat_cols_to_impute = X_train.select_dtypes(include=['object']).columns
mode_imputer = SimpleImputer(strategy='most_frequent')

X_train[cat_cols_to_impute] = mode_imputer.fit_transform(X_train[cat_cols_to_impute])
X_test[cat_cols_to_impute] = mode_imputer.transform(X_test[cat_cols_to_impute])

### 2.4 Evidência do Tratamento
Verificação final para atestar que não restaram valores nulos nas bases de dados após a imputação.

In [ ]:
# Contagem de nulos antes e depois
missing_before = X_train_before.isnull().sum().sum()
missing_after = X_train.isnull().sum().sum()

print(f"Total de valores ausentes ANTES do tratamento: {missing_before}")
print(f"Total de valores ausentes DEPOIS do tratamento: {missing_after}")

# Evidência prática em uma coluna importante (EXT_SOURCE_3)
print("\n--- Estatísticas de EXT_SOURCE_3 ANTES ---")
print(X_train_before['EXT_SOURCE_3'].describe()[['count', 'mean', 'std']])
print("\n--- Estatísticas de EXT_SOURCE_3 DEPOIS ---")
print(X_train['EXT_SOURCE_3'].describe()[['count', 'mean', 'std']])